# Story day 1-20 analysis

In [1]:
# hide-output
# Import libraries and initialise the BigQuery connector

# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

# Standard data analysis stack + project utilities
import pandas as pd
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html
import plotly.express as px
import numpy as np
import plotly.graph_objects as go

bqc = BigQueryConnector()

<!-- hide --> 
## Aux functions

In [2]:
# hide-output
# Helper functions for weighted progression, percentile calculation, level visualisations, and retention significance
from aux_functions import (
    compute_weighted_progression,
    weighted_quantiles,
    add_event_annotations,
    add_median_lines,
    plot_percentile_comparison,
    compute_retention_significance,
    plot_retention_significance,
)

## Get data

### Player level and game day

In [3]:
# Compute symmetric A/B date windows of equal length anchored on 2026-06-01 (FTUE launch date)

import datetime as dt

new_ftue_date = dt.datetime(2026, 7, 1)
#days_from_start = (dt.datetime.today() - new_ftue_date).days
days_from_start = 21  # Set the number of days for the first window
start_date1 = new_ftue_date - dt.timedelta(days=days_from_start)
end_date1 = new_ftue_date-dt.timedelta(days=1)
start_date2 = new_ftue_date
#end_date2 = dt.datetime.today()-dt.timedelta(days=1)
end_date2 = new_ftue_date + dt.timedelta(days=days_from_start-1)

# print all dates
print(f"Start Date 1: {start_date1.strftime('%Y-%m-%d')}")
print(f"End Date 1: {end_date1.strftime('%Y-%m-%d')}")
print(f"Start Date 2: {start_date2.strftime('%Y-%m-%d')}")
print(f"End Date 2: {end_date2.strftime('%Y-%m-%d')}")

Start Date 1: 2026-06-10
End Date 1: 2026-06-30
Start Date 2: 2026-07-01
End Date 2: 2026-07-21


In [4]:
# hide-output
# Toggle: True re-queries BigQuery and overwrites local cache, False loads from pickle
refresh_data = True

In [5]:
# hide-output
# Estimate query cost for player level + game day SQL before executing

query_location = './sql/playerlevel.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['CPE']        
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 16.32 GB when run.
Estimated query cost: $0.11


In [6]:
# hide-output
# Fetch player level + game day data from BigQuery or load from local pickle cache
data = pd.DataFrame()

# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    data = bqc.get(query='./sql/playerlevel.sql', is_path=True, query_parameters=parameters)
    data.to_pickle('./data/playprogression.pkl')
else:
    # Load from local cache to avoid repeated query costs
    data = pd.read_pickle('./data/playprogression.pkl')

In [7]:
# hide-output
# Preview raw player level and game day data
data

,user_id,dt,install_dt,country_code,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,platform,display_campaign_network,acquisition_type,install_build_version
0,CADDF58FBA4A8999,2026-07-21,2026-07-21,ZA,2026-07-19,2026-07-01,0,1,1,IOS,Non-Attributed,Non-Attributed,0.80.0
1,FB548924AC1DA0DA,2026-07-21,2026-07-21,BA,2026-07-19,2026-07-01,0,4,2,IOS,Non-Attributed,Non-Attributed,0.80.0
2,550DAACEC841D923,2026-07-21,2026-07-21,IT,2026-07-19,2026-07-01,0,2,1,AND,Non-Attributed,Non-Attributed,0.80.0
3,68E935895806E10C,2026-07-21,2026-07-21,US,2026-07-19,2026-07-01,0,4,2,IOS,FACEBOOK,UA,0.80.0
4,69941217A4047429,2026-07-21,2026-07-21,DE,2026-07-19,2026-07-01,0,3,1,IOS,FACEBOOK,UA,0.80.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
107725,27B0A2BCE3F44604,2026-07-20,2026-07-20,IN,2026-07-19,2026-07-01,0,2,1,IOS,Non-Attributed,Non-Attributed,0.79.0
107726,CC89DAA05F01EC87,2026-07-20,2026-07-20,US,2026-07-19,2026-07-01,0,2,1,AND,Non-Attributed,Non-Attributed,0.79.0
107727,13E91F111D8CA42B,2026-07-21,2026-07-20,ZA,2026-07-19,2026-07-01,1,4,2,AND,Non-Attributed,Non-Attributed,0.79.0
107728,4B0E8CF3319E0F11,2026-07-20,2026-07-20,SK,2026-07-19,2026-07-01,0,2,1,IOS,Non-Attributed,Non-Attributed,0.80.0


<!-- hide --> 
### Retention

In [8]:
# hide-output
# Estimate query cost for per-install-date retention SQL
query_location = './sql/retention.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['CPE']        
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 1.15 GB when run.
Estimated query cost: $0.01


In [9]:
# hide-output
# Fetch per-install-date retention data from BigQuery or load from local pickle cache
retention_data = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    retention_data = bqc.get(query='./sql/retention.sql', is_path=True, query_parameters=parameters)
    retention_data.to_pickle('./data/retention.pkl')
else:
    # Load from local cache to avoid repeated query costs
    retention_data = pd.read_pickle('./data/retention.pkl')

In [10]:
# hide-output
# Sort retention data and spot-check Android rows
retention_data.sort_values(['install_dt', 'dx','platform'], inplace=True)
retention_data[retention_data['platform'] == 'AND']

,install_dt,dx,platform,cohort_size,retained_size,retention_rate
0,2026-06-10,0,AND,305,305,1.000000
2,2026-06-10,1,AND,305,80,0.262295
5,2026-06-10,3,AND,305,48,0.157377
6,2026-06-10,7,AND,305,41,0.134426
8,2026-06-10,14,AND,305,38,0.124590
...,...,...,...,...,...,...
427,2026-07-19,0,AND,273,273,1.000000
428,2026-07-19,1,AND,273,62,0.227106
431,2026-07-20,0,AND,235,235,1.000000
433,2026-07-20,1,AND,235,74,0.314894


In [51]:
# hide-output
# Estimate query cost for FTUE-split retention SQL (all users)
query_location = './sql/retentiontotal.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['CPE']        
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 1.15 GB when run.
Estimated query cost: $0.01


In [52]:
# hide-output
# Fetch FTUE-split retention (all users) from BigQuery or load from local pickle cache
retention_data_total = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    retention_data_total = bqc.get(query='./sql/retentiontotal.sql', is_path=True, query_parameters=parameters)
    retention_data_total.to_pickle('./data/retentiontotal.pkl')
else:
    # Load from local cache to avoid repeated query costs
    retention_data_total = pd.read_pickle('./data/retentiontotal.pkl')

In [55]:
# hide-output
# Sort FTUE retention data and spot-check at D14
retention_data_total.sort_values([ 'dx', 'platform','FTUE_flag'], inplace=True)
#retention_data_total[retention_data_total['dx'] == 14]
retention_data_total

,dx,platform,FTUE_flag,num_cohorts,cohort_size,retained_size,retention_rate
1,0,AND,A.Pre-FTUE revamp,21,6039,6039,1.000000
0,0,AND,B.Post-FTUE revamp,21,6077,6077,1.000000
3,0,IOS,A.Pre-FTUE revamp,21,13153,13153,1.000000
2,0,IOS,B.Post-FTUE revamp,21,13710,13710,1.000000
4,1,AND,A.Pre-FTUE revamp,20,4659,1296,0.278171
5,1,AND,B.Post-FTUE revamp,20,4888,1374,0.281097
6,1,IOS,A.Pre-FTUE revamp,20,12526,3946,0.315025
7,1,IOS,B.Post-FTUE revamp,20,13017,4209,0.323346
8,3,AND,A.Pre-FTUE revamp,18,4197,699,0.166548
9,3,AND,B.Post-FTUE revamp,18,4369,773,0.176928


In [14]:
# hide-output
# Estimate query cost for FTUE-split organic-only retention SQL
query_location = './sql/retentiontotalNA.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['CPE']        
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 1.15 GB when run.
Estimated query cost: $0.01


In [15]:
# hide-output
# Fetch FTUE-split organic-only retention from BigQuery or load from local pickle cache
retention_data_total_na = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache
if refresh_data:
    retention_data_total_na = bqc.get(query='./sql/retentiontotalNA.sql', is_path=True, query_parameters=parameters)
    retention_data_total_na.to_pickle('./data/retentiontotalNA.pkl')
else:
    # Load from local cache to avoid repeated query costs
    retention_data_total_na = pd.read_pickle('./data/retentiontotalNA.pkl')

In [16]:
# hide-output
# Sort and preview organic-only FTUE retention data
retention_data_total_na.sort_values([ 'dx', 'platform','FTUE_flag'], inplace=True)
retention_data_total_na

,dx,platform,FTUE_flag,num_cohorts,cohort_size,retained_size,retention_rate
1,0,AND,A.Pre-FTUE revamp,21,5236,5236,1.000000
0,0,AND,B.Post-FTUE revamp,21,5413,5413,1.000000
3,0,IOS,A.Pre-FTUE revamp,21,10962,10962,1.000000
2,0,IOS,B.Post-FTUE revamp,21,11899,11899,1.000000
4,1,AND,A.Pre-FTUE revamp,20,4016,1215,0.302540
5,1,AND,B.Post-FTUE revamp,20,4331,1295,0.299007
6,1,IOS,A.Pre-FTUE revamp,20,10412,3304,0.317326
7,1,IOS,B.Post-FTUE revamp,20,11299,3643,0.322418
8,3,AND,A.Pre-FTUE revamp,18,3600,651,0.180833
9,3,AND,B.Post-FTUE revamp,18,3846,718,0.186687


### AB metrics

In [17]:
# hide-output
# Estimate query cost for FTUE-split organic-only retention SQL
query_location = './sql/abmetrics.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),  
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 2.81 GB when run.
Estimated query cost: $0.02


In [18]:
# hide-output
# Fetch FTUE-split organic-only retention from BigQuery or load from local pickle cache
abmetrics = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache
if refresh_data:
    abmetrics = bqc.get(query='./sql/abmetrics.sql', is_path=True, query_parameters=parameters)
    abmetrics.to_pickle('./data/abmetrics.pkl')
else:
    # Load from local cache to avoid repeated query costs
    abmetrics = pd.read_pickle('./data/abmetrics.pkl')

In [19]:
abmetrics

,user_id,install_dt,dt,days_since_install,max_level,max_gameday,active,active_cumu,days_since_last_active_exc_today,n_sessions,...,n_cashdash_ms_completed,n_datedash_ms_completed,n_space_ms_completed,n_art_ms_completed,n_roadtrip_ms_completed,n_fortunes_ms_completed,n_timedalbum_sets_completed,reached_600_clash_points,reached_rank1,loading_timestamp
0,79170A9D6C349201,2026-07-13,2026-07-13,0,2,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-07-18 04:02:03.876910+00:00
1,E9110B9D17760D3D,2026-07-13,2026-07-13,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-07-18 04:02:03.876910+00:00
2,4C39A01115AE5545,2026-07-13,2026-07-13,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-07-18 04:02:03.876910+00:00
3,31DF31117210B210,2026-07-13,2026-07-13,0,1,1,1,1,<NA>,2,...,0,0,0,0,0,0,0,0,0,2026-07-18 04:02:03.876910+00:00
4,195A0F4BDAB19A38,2026-07-13,2026-07-13,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-07-18 04:02:03.876910+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175916,FF8A6DA63262AC8D,2026-06-27,2026-06-30,3,17,12,1,4,1,3,...,0,0,0,0,7,0,1,1,0,2026-07-05 02:58:50.479379+00:00
175917,5582BA2285A9A4D7,2026-06-15,2026-06-30,15,28,24,1,15,1,3,...,0,0,0,0,28,0,1,1,0,2026-07-05 02:58:50.479379+00:00
175918,1B307E86B4F5D38F,2026-06-26,2026-06-30,4,22,18,1,5,1,13,...,0,0,0,0,26,0,7,1,1,2026-07-05 02:58:50.479379+00:00
175919,262FBE1B77137379,2026-06-29,2026-06-30,1,17,13,1,2,1,7,...,0,0,0,0,14,0,0,1,0,2026-07-05 02:58:50.479379+00:00


<!-- hide --> 
## Process data

In [56]:
# hide-output
# Assign FTUE flag, cap days_since_install to match B.new window, and drop immature cohort rows
dt_mode = 'install_dt'
processed_data = data.copy()

processed_data['install_dt'] = processed_data[dt_mode]

processed_data['CPE_flag'] = ['Y' if x == 'CPE' else 'N' for x in processed_data['acquisition_type']]
processed_data['FTUE_flag'] = ['B.new' if x >= new_ftue_date else 'A.old' for x in pd.to_datetime(processed_data['install_dt'])]

# Making comparison fair by capping the days_since_install for B.new to match the number of days since the new FTUE date.
max_dayx_B_new = (pd.to_datetime('today') - pd.to_datetime(new_ftue_date)).days
processed_data = processed_data[~(processed_data['days_since_install'] > max_dayx_B_new)]

# Cap data to only gameday 20 or less for both cohorts to ensure fair comparison
processed_data = processed_data[processed_data['days_since_install'] <= 20]

processed_data.loc[:,'dummy'] = 'dummy'

# Require each cohort to have had enough calendar time to be meaningful:
# weekly cohorts need 7 days, monthly cohorts need 30.
min_days_since_install = 0
if dt_mode == 'install_dt_week':
    min_days_since_install = 7
elif dt_mode == 'install_dt_month':
    min_days_since_install = 30 

# Drop rows where the player's most recent observed day falls within the cohort's maturity window.
# This prevents partially-observed cohorts from pulling down progression averages.
processed_data = processed_data[processed_data['days_since_install'] <= (pd.to_datetime('today') - pd.to_datetime(processed_data['install_dt'])).dt.days - min_days_since_install]


# Add a country filter just to see if metrics align better with the US-only data. This is a temporary filter for testing purposes.
#processed_data = processed_data[processed_data['country_code'] == 'US']

processed_data

,user_id,dt,install_dt,country_code,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,platform,display_campaign_network,acquisition_type,install_build_version,CPE_flag,FTUE_flag,dummy
61871,100186873D03CE09,2026-06-28,2026-06-28,US,2026-06-28,2026-06-01,0,5,3,IOS,Non-Attributed,Non-Attributed,0.78.0,N,A.old,dummy
30611,1008C486C3810433,2026-06-18,2026-06-18,PH,2026-06-14,2026-06-01,0,6,4,IOS,Non-Attributed,Non-Attributed,0.77.0,N,A.old,dummy
30432,1008C486C3810433,2026-06-19,2026-06-18,PH,2026-06-14,2026-06-01,1,8,5,IOS,Non-Attributed,Non-Attributed,0.77.0,N,A.old,dummy
28163,1008C486C3810433,2026-06-20,2026-06-18,PH,2026-06-14,2026-06-01,2,10,7,IOS,Non-Attributed,Non-Attributed,0.77.0,N,A.old,dummy
28898,1008C486C3810433,2026-06-21,2026-06-18,PH,2026-06-14,2026-06-01,3,11,8,IOS,Non-Attributed,Non-Attributed,0.77.0,N,A.old,dummy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44306,FFF7CA7736D2BF38,2026-06-22,2026-06-22,US,2026-06-21,2026-06-01,0,2,1,IOS,FACEBOOK,UA,0.78.0,N,A.old,dummy
43533,FFF7CA7736D2BF38,2026-06-23,2026-06-22,US,2026-06-21,2026-06-01,1,6,4,IOS,FACEBOOK,UA,0.78.0,N,A.old,dummy
97756,FFFC66E056EBEBDF,2026-07-13,2026-07-13,DE,2026-07-12,2026-07-01,0,1,1,AND,Non-Attributed,Non-Attributed,0.79.0,N,B.new,dummy
97197,FFFC66E056EBEBDF,2026-07-14,2026-07-13,DE,2026-07-12,2026-07-01,1,1,1,AND,Non-Attributed,Non-Attributed,0.79.0,N,B.new,dummy


In [57]:
# hide-output
# Sanity-check unique user counts per FTUE group
test = processed_data.groupby(['FTUE_flag']).agg(
    users=('user_id', 'nunique')
).reset_index()

test

,FTUE_flag,users
0,A.old,11779
1,B.new,11809


## Retention

In [58]:
# hide-output
# Add combined dx_platform column for the per-install-date retention line chart
retention_data['combined_dimension'] = retention_data['dx'].astype(str) + '_' + retention_data['platform']
retention_data

,install_dt,dx,platform,cohort_size,retained_size,retention_rate,combined_dimension
0,2026-06-10,0,AND,305,305,1.000000,0_AND
1,2026-06-10,0,IOS,573,573,1.000000,0_IOS
2,2026-06-10,1,AND,305,80,0.262295,1_AND
3,2026-06-10,1,IOS,573,178,0.310646,1_IOS
5,2026-06-10,3,AND,305,48,0.157377,3_AND
...,...,...,...,...,...,...,...
430,2026-07-20,0,IOS,558,558,1.000000,0_IOS
433,2026-07-20,1,AND,235,74,0.314894,1_AND
432,2026-07-20,1,IOS,558,192,0.344086,1_IOS
434,2026-07-21,0,AND,221,221,1.000000,0_AND


In [59]:
# Per-install-date retention rate over time by dx/platform (figure disabled — used for investigation only)
fig = px.line(retention_data[retention_data['dx'] !=0], 
              x='install_dt', 
              y='retention_rate',
              color='combined_dimension',
              title='Retention rate',
              facet_row='platform',
              width=1200,
              height=800,
              hover_data={'install_dt': True, 'retained_size': True},)

#fig.show()

In [60]:
# hide-output
# Preview FTUE-split retention totals (all users)
retention_data_total

,dx,platform,FTUE_flag,num_cohorts,cohort_size,retained_size,retention_rate
1,0,AND,A.Pre-FTUE revamp,21,6039,6039,1.000000
0,0,AND,B.Post-FTUE revamp,21,6077,6077,1.000000
3,0,IOS,A.Pre-FTUE revamp,21,13153,13153,1.000000
2,0,IOS,B.Post-FTUE revamp,21,13710,13710,1.000000
4,1,AND,A.Pre-FTUE revamp,20,4659,1296,0.278171
5,1,AND,B.Post-FTUE revamp,20,4888,1374,0.281097
6,1,IOS,A.Pre-FTUE revamp,20,12526,3946,0.315025
7,1,IOS,B.Post-FTUE revamp,20,13017,4209,0.323346
8,3,AND,A.Pre-FTUE revamp,18,4197,699,0.166548
9,3,AND,B.Post-FTUE revamp,18,4369,773,0.176928


### Cohort sizes

In [61]:
# Bar chart: cohort sizes at each retention checkpoint by FTUE group and platform
df_plot = retention_data_total[retention_data_total['dx'] != 0].copy()

# Convert dx to ordered categorical for proper x-axis ordering
dx_vals = sorted(df_plot['dx'].unique())
dx_labels = ['D' + str(d) for d in dx_vals]
df_plot['dx_cat'] = pd.Categorical(
    'D' + df_plot['dx'].astype(str),
    categories=dx_labels,
    ordered=True
)

fig = px.bar(
    df_plot,
    x='dx_cat',
    y='cohort_size',
    color='FTUE_flag',
    text='cohort_size',
    title='Cohort sizes',
    facet_row='platform',
    barmode='group',
    width=1200,
    height=900,
    hover_data={'retained_size': True, 'cohort_size': True, 'num_cohorts': True},
    category_orders={'dx_cat': dx_labels}
)

fig.update_traces(texttemplate='%{text:0}', textposition='outside')
fig.update_layout(
    #yaxis_tickformat='.0%',
    #yaxis2_tickformat='.0%',
    uniformtext_minsize=8,
    uniformtext_mode='hide'
)

fig.show()

### Retention rate 

In [62]:
# Bar chart: D1–D21 retention rates by FTUE group and platform (all users)
# Each Dx is tested independently with a two-proportion z-test; Wilson 95% CIs shown as error bars
dummy = plot_retention_significance(
    retention_data_total,
    title='Retention rate by FTUE group',
)

### Retention rate (Organics only)

In [63]:
# hide-output
# Bar chart: D1–D21 retention rates by FTUE group and platform (organic / non-attributed only)
# Each Dx is tested independently; small D21 organic cohorts annotated with users needed for significance
dummy = plot_retention_significance(
    retention_data_total_na,
    title='Retention rate by FTUE group — organic only',
)

## Player max level distribution

In [64]:
# hide-output
# Preview player data
processed_data

,user_id,dt,install_dt,country_code,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,platform,display_campaign_network,acquisition_type,install_build_version,CPE_flag,FTUE_flag,dummy
61871,100186873D03CE09,2026-06-28,2026-06-28,US,2026-06-28,2026-06-01,0,5,3,IOS,Non-Attributed,Non-Attributed,0.78.0,N,A.old,dummy
30611,1008C486C3810433,2026-06-18,2026-06-18,PH,2026-06-14,2026-06-01,0,6,4,IOS,Non-Attributed,Non-Attributed,0.77.0,N,A.old,dummy
30432,1008C486C3810433,2026-06-19,2026-06-18,PH,2026-06-14,2026-06-01,1,8,5,IOS,Non-Attributed,Non-Attributed,0.77.0,N,A.old,dummy
28163,1008C486C3810433,2026-06-20,2026-06-18,PH,2026-06-14,2026-06-01,2,10,7,IOS,Non-Attributed,Non-Attributed,0.77.0,N,A.old,dummy
28898,1008C486C3810433,2026-06-21,2026-06-18,PH,2026-06-14,2026-06-01,3,11,8,IOS,Non-Attributed,Non-Attributed,0.77.0,N,A.old,dummy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44306,FFF7CA7736D2BF38,2026-06-22,2026-06-22,US,2026-06-21,2026-06-01,0,2,1,IOS,FACEBOOK,UA,0.78.0,N,A.old,dummy
43533,FFF7CA7736D2BF38,2026-06-23,2026-06-22,US,2026-06-21,2026-06-01,1,6,4,IOS,FACEBOOK,UA,0.78.0,N,A.old,dummy
97756,FFFC66E056EBEBDF,2026-07-13,2026-07-13,DE,2026-07-12,2026-07-01,0,1,1,AND,Non-Attributed,Non-Attributed,0.79.0,N,B.new,dummy
97197,FFFC66E056EBEBDF,2026-07-14,2026-07-13,DE,2026-07-12,2026-07-01,1,1,1,AND,Non-Attributed,Non-Attributed,0.79.0,N,B.new,dummy


In [94]:
# hide-output
# Build level funnel with P10/P50/P90 percentiles per FTUE group, platform, and day since install
days_since_install_limit = 21

pl_ftue_funnel_agg = processed_data.groupby(['max_gameday','FTUE_flag','platform', 'days_since_install']).agg(
    users=('user_id', 'nunique')
).reset_index()

pl_ftue_funnel_total_agg = pl_ftue_funnel_agg.groupby(['FTUE_flag','platform','days_since_install']).agg(
    total_users=('users', 'sum')
).reset_index()

pl_ftue_funnel_agg = pl_ftue_funnel_agg.merge(pl_ftue_funnel_total_agg, on=['FTUE_flag','platform','days_since_install'])
pl_ftue_funnel_agg['pctg_users'] = pl_ftue_funnel_agg['users'] / pl_ftue_funnel_agg['total_users']


pl_ftue_funnel_agg['pctg_diff_users'] = pl_ftue_funnel_agg.groupby(['max_gameday','platform','days_since_install'])['pctg_users'].pct_change().fillna(0)

pl_ftue_funnel_agg['combined_dimension'] = pl_ftue_funnel_agg['FTUE_flag'].astype(str) + ' | ' + pl_ftue_funnel_agg['days_since_install'].astype(str)

# Calculate percentiles by FTUE_flag, platform, and days_since_install
level_dist = processed_data.groupby(['days_since_install', 'max_gameday', 'FTUE_flag', 'platform']).agg(
    users=('user_id', 'count')
).reset_index()

level_pcts_by_group = level_dist[level_dist.days_since_install<=days_since_install_limit].groupby(['days_since_install', 'FTUE_flag', 'platform']).apply(
    weighted_quantiles, measure_col='max_gameday', include_groups=False
).reset_index()

# Rename columns for clarity
#level_pcts_by_group = level_pcts_by_group.rename(columns={'p10': 'p10_max_gameday', 'p50': 'p50_max_gameday', 'p90': 'p90_max_gameday'})


# Merge into pl_ftue_funnel_agg
pl_ftue_funnel_agg = pl_ftue_funnel_agg.merge(
    level_pcts_by_group, 
    on=['days_since_install', 'FTUE_flag', 'platform'], 
    how='left'
)

pl_ftue_funnel_agg = pl_ftue_funnel_agg.loc[pl_ftue_funnel_agg['days_since_install'].isin([0,1,3,7,14,21,28])]


pl_ftue_funnel_agg

,max_gameday,FTUE_flag,platform,days_since_install,users,total_users,pctg_users,pctg_diff_users,combined_dimension,p10,p50,p90
0,1,A.old,AND,0,1275,2751,0.463468,0.0,A.old | 0,1,2,3
1,1,A.old,AND,1,142,1162,0.122203,0.0,A.old | 1,1,3,4
3,1,A.old,AND,3,42,745,0.056376,0.0,A.old | 3,2,4,8
7,1,A.old,AND,7,15,593,0.025295,0.0,A.old | 7,3,7,11
14,1,A.old,AND,14,6,466,0.012876,0.0,A.old | 14,4,10,16
...,...,...,...,...,...,...,...,...,...,...,...,...
2448,205,B.new,AND,1,1,1235,0.000810,0.0,B.new | 1,1,3,4
2450,205,B.new,AND,3,1,721,0.001387,0.0,B.new | 3,2,4,8
2451,205,B.new,AND,7,1,411,0.002433,0.0,B.new | 7,3,7,10
2458,205,B.new,AND,14,2,157,0.012739,0.0,B.new | 14,4,10,16


In [95]:
# hide-output
# Define level milestone annotations for A.old and B.new FTUE feature unlock points
events_config = {
    #'A.old': [
    #    {'level': 7, 'name': 'SP', 'color':'blue'},
    #    {'level': 8, 'name': 'Deco', 'color': 'blue'},
    #    {'level': 10, 'name': 'TimedC', 'color': 'blue'},
    #    {'level': 16, 'name': 'TA', 'color': 'blue'},
    #    {'level': 20, 'name': 'GenB', 'color': 'blue'},
    #   {'level': 25, 'name': 'TgtEvt', 'color': 'blue'},
    #],
    'B.new': [
        {'level': 6, 'name': 'SP', 'color': 'red'},
        {'level': 9, 'name': 'TASign', 'color': 'red'},
        {'level': 10, 'name': 'Deco', 'color': 'red'},
        {'level': 12, 'name': 'TA', 'color': 'red'},
        {'level': 14, 'name': 'GenB', 'color': 'red'},
        {'level': 17, 'name': 'TimedC', 'color': 'red'},
        {'level': 20, 'name': 'TgtEvt', 'color': 'red'},
    ]
}

In [96]:
max_gameday = 14

# Level distribution charts: user counts, percentage share, and day-over-day diff (up to level 30)
fig = px.line(pl_ftue_funnel_agg[pl_ftue_funnel_agg['max_gameday'] <= max_gameday], 
              x='max_gameday', 
              labels={'max_gameday': 'Player Game Day'},
              y='users',
              color='combined_dimension',
              title='Players game day distribution',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'max_gameday': True, 'users': True},)

#fig = add_event_annotations(fig, events_config, x_col='max_gameday', ftue_col='FTUE_flag', data=pl_ftue_funnel_agg, show_annotations=False)
#fig = add_median_lines(fig, pl_ftue_funnel_agg, x_col='p50_max_gameday', ftue_col='FTUE_flag', platform_col='platform')

fig.show()

# hide-output
fig = px.line(pl_ftue_funnel_agg[pl_ftue_funnel_agg['max_gameday'] <= max_gameday], 
              x='max_gameday', 
              labels={'max_gameday': 'Player Game Day'},
              y='pctg_users',
              color='combined_dimension',
              title='Players at each game day (percentage)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'max_gameday': True, 'users': True},)

#fig = add_event_annotations(fig, events_config, x_col='max_gameday', ftue_col='FTUE_flag', data=pl_ftue_funnel_agg, show_annotations=True)

fig.show()

# hide-output
fig = px.line(pl_ftue_funnel_agg[pl_ftue_funnel_agg['max_gameday'] <= max_gameday], 
              x='max_gameday', 
              labels={'max_gameday': 'Player Game Day'},
              y='pctg_diff_users',
              color='combined_dimension',
              title='Players at each game day (percentage diff change)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'max_gameday': True, 'users': True},)

#fig = add_event_annotations(fig, events_config, x_col='max_gameday', ftue_col='FTUE_flag', data=pl_ftue_funnel_agg, show_annotations=False)

fig.show()

### Percentile max level reached

In [97]:
def plot_percentile_comparison(df, percentile='all', measure_name='Max Game Day'):
    """
    Plot percentile comparison between FTUE groups.
    
    Args:
        df: DataFrame with columns: days_since_install, FTUE_flag, platform, p10, p50, p90
        percentile: 'all', 'p10', 'p50', or 'p90'
        measure_name: Name of the measure for axis labels (default: 'Max Game Day')
    """
    # Rename columns to generic percentile names if they have suffixes
    df_plot = df.copy()
    for col in df_plot.columns:
        if col.startswith('p') and col[1:].replace('0', '').isdigit():
            # Already in generic format (p10, p50, p90)
            pass
        elif '_' in col and any(col.startswith(f'p{x}_') for x in ['10', '50', '90']):
            # Has suffix like p10_max_gameday -> rename to p10
            percentile_num = col.split('_')[0]
            df_plot = df_plot.rename(columns={col: percentile_num})
    
    if percentile == 'all':
        percentile_cols = ['p10', 'p50', 'p90']
    else:
        percentile_cols = [percentile]
    
    df_melted = df_plot.melt(
        id_vars=['days_since_install', 'FTUE_flag', 'platform'],
        value_vars=percentile_cols,
        var_name='percentile',
        value_name='value'
    )
    
    fig = px.line(
        df_melted,
        x='days_since_install',
        y='value',
        color='FTUE_flag',
        facet_col='platform',
        line_dash='percentile',
        title=f'Percentile comparison - {percentile if percentile != "all" else "P10/P50/P90"}',
        width=1500,
        height=600,
        labels={'value': measure_name, 'days_since_install': 'Days Since Install'}
    )
    
    fig.show()

In [98]:
# Band chart: P10/P50/P90 level progression comparison between A.old and B.new
plot_percentile_comparison(level_pcts_by_group, percentile='all', measure_name='Max Game Day')

# To inspect a single percentile with diff bar:
# plot_percentile_comparison(level_pcts_by_group, percentile='p50')
# plot_percentile_comparison(level_pcts_by_group, percentile='p10')
# plot_percentile_comparison(level_pcts_by_group, percentile='p90')

### Average max level reached

In [106]:
# hide-output
# Compute weighted average max level by FTUE group, platform, and days since install
days_since_install_baseline = 28

data_filtered = processed_data[processed_data['days_since_install'] <= days_since_install_baseline]

pl_ftue_max_level_agg = compute_weighted_progression(data_filtered, measure_col='max_gameday', dimension_cols=['dummy', 'days_since_install','FTUE_flag','platform'], min_bucket_size=10)
pl_ftue_max_level_agg['combined_dimension'] = pl_ftue_max_level_agg['dummy'].astype(str) + ' | ' + pl_ftue_max_level_agg['FTUE_flag']

pl_ftue_max_level_agg['pctg_diff_max_gameday'] = pl_ftue_max_level_agg.groupby(['days_since_install', 'platform'])['weighted_avg_max_gameday'].pct_change().fillna(0)

pl_ftue_total_users = pl_ftue_max_level_agg[pl_ftue_max_level_agg['days_since_install'] == 0][['FTUE_flag', 'platform', 'cohort_users']].rename(columns={'cohort_users': 'total_cohort_users'})
pl_ftue_max_level_agg = pl_ftue_max_level_agg.merge(pl_ftue_total_users, on=['FTUE_flag','platform'], how='left')
pl_ftue_max_level_agg['pctg_users'] = pl_ftue_max_level_agg['cohort_users'] / pl_ftue_max_level_agg['total_cohort_users']
pl_ftue_max_level_agg['pctg_diff_users'] = pl_ftue_max_level_agg.groupby(['days_since_install', 'platform'])['pctg_users'].pct_change().fillna(0)

pl_ftue_max_level_agg

,dummy,days_since_install,FTUE_flag,platform,cohort_users,weighted_avg_max_gameday,combined_dimension,pctg_diff_max_gameday,total_cohort_users,pctg_users,pctg_diff_users
0,dummy,0,A.old,AND,2735,1.901490,dummy | A.old,0.000000,2735,1.000000,0.00000
1,dummy,0,A.old,IOS,8861,1.820995,dummy | A.old,0.000000,8861,1.000000,0.00000
2,dummy,0,B.new,AND,2696,2.065386,dummy | B.new,0.086193,2696,1.000000,0.00000
3,dummy,0,B.new,IOS,8992,1.979925,dummy | B.new,0.087276,8992,1.000000,0.00000
4,dummy,1,A.old,AND,1138,3.703959,dummy | A.old,0.000000,2735,0.416088,0.00000
...,...,...,...,...,...,...,...,...,...,...,...
76,dummy,19,A.old,AND,357,18.087167,dummy | A.old,0.000000,2735,0.130530,0.00000
77,dummy,19,A.old,IOS,958,14.129225,dummy | A.old,0.000000,8861,0.108114,0.00000
78,dummy,19,B.new,IOS,22,14.569767,dummy | B.new,0.031180,8992,0.002447,-0.97737
79,dummy,20,A.old,AND,351,19.181384,dummy | A.old,0.000000,2735,0.128336,0.00000


In [107]:
# hide-output
# Bar chart: surviving cohort size at each day since install by FTUE group
fig = px.bar(pl_ftue_max_level_agg, 
              x='days_since_install', 
              y='cohort_users',
              color='combined_dimension',
              title='Players at each day since install',
              facet_row='platform',
              width=1200,
              height=800,
              barmode='group',
              hover_data={'weighted_avg_max_gameday': True, 'cohort_users': True},)

fig.show()

In [108]:
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(pl_ftue_max_level_agg, 
              x='days_since_install', 
              y='weighted_avg_max_gameday',
              color='combined_dimension',
              title='Player level reached at day x since install',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'weighted_avg_max_gameday': True, 'cohort_users': True},)

fig.show()

# hide-output
fig = px.line(pl_ftue_max_level_agg, 
              x='days_since_install', 
              y='pctg_diff_max_gameday',
              color='combined_dimension',
              title='Player level reached at day x since install (pct change)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'weighted_avg_max_gameday': True, 'cohort_users': True},)

fig.show()

In [72]:
# hide-output
# Sort data by user and day for per-user progression analysis
data.sort_values(['user_id', 'days_since_install'], inplace=True)
data

,user_id,dt,install_dt,country_code,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,platform,display_campaign_network,acquisition_type,install_build_version
61871,100186873D03CE09,2026-06-28,2026-06-28,US,2026-06-28,2026-06-01,0,5,3,IOS,Non-Attributed,Non-Attributed,0.78.0
30611,1008C486C3810433,2026-06-18,2026-06-18,PH,2026-06-14,2026-06-01,0,6,4,IOS,Non-Attributed,Non-Attributed,0.77.0
30432,1008C486C3810433,2026-06-19,2026-06-18,PH,2026-06-14,2026-06-01,1,8,5,IOS,Non-Attributed,Non-Attributed,0.77.0
28163,1008C486C3810433,2026-06-20,2026-06-18,PH,2026-06-14,2026-06-01,2,10,7,IOS,Non-Attributed,Non-Attributed,0.77.0
28898,1008C486C3810433,2026-06-21,2026-06-18,PH,2026-06-14,2026-06-01,3,11,8,IOS,Non-Attributed,Non-Attributed,0.77.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
44306,FFF7CA7736D2BF38,2026-06-22,2026-06-22,US,2026-06-21,2026-06-01,0,2,1,IOS,FACEBOOK,UA,0.78.0
43533,FFF7CA7736D2BF38,2026-06-23,2026-06-22,US,2026-06-21,2026-06-01,1,6,4,IOS,FACEBOOK,UA,0.78.0
97756,FFFC66E056EBEBDF,2026-07-13,2026-07-13,DE,2026-07-12,2026-07-01,0,1,1,AND,Non-Attributed,Non-Attributed,0.79.0
97197,FFFC66E056EBEBDF,2026-07-14,2026-07-13,DE,2026-07-12,2026-07-01,1,1,1,AND,Non-Attributed,Non-Attributed,0.79.0


## Engagement metrics

In [73]:
abmetrics

,user_id,install_dt,dt,days_since_install,max_level,max_gameday,active,active_cumu,days_since_last_active_exc_today,n_sessions,...,n_cashdash_ms_completed,n_datedash_ms_completed,n_space_ms_completed,n_art_ms_completed,n_roadtrip_ms_completed,n_fortunes_ms_completed,n_timedalbum_sets_completed,reached_600_clash_points,reached_rank1,loading_timestamp
0,79170A9D6C349201,2026-07-13,2026-07-13,0,2,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-07-18 04:02:03.876910+00:00
1,E9110B9D17760D3D,2026-07-13,2026-07-13,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-07-18 04:02:03.876910+00:00
2,4C39A01115AE5545,2026-07-13,2026-07-13,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-07-18 04:02:03.876910+00:00
3,31DF31117210B210,2026-07-13,2026-07-13,0,1,1,1,1,<NA>,2,...,0,0,0,0,0,0,0,0,0,2026-07-18 04:02:03.876910+00:00
4,195A0F4BDAB19A38,2026-07-13,2026-07-13,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-07-18 04:02:03.876910+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175916,FF8A6DA63262AC8D,2026-06-27,2026-06-30,3,17,12,1,4,1,3,...,0,0,0,0,7,0,1,1,0,2026-07-05 02:58:50.479379+00:00
175917,5582BA2285A9A4D7,2026-06-15,2026-06-30,15,28,24,1,15,1,3,...,0,0,0,0,28,0,1,1,0,2026-07-05 02:58:50.479379+00:00
175918,1B307E86B4F5D38F,2026-06-26,2026-06-30,4,22,18,1,5,1,13,...,0,0,0,0,26,0,7,1,1,2026-07-05 02:58:50.479379+00:00
175919,262FBE1B77137379,2026-06-29,2026-06-30,1,17,13,1,2,1,7,...,0,0,0,0,14,0,0,1,0,2026-07-05 02:58:50.479379+00:00


In [74]:
processed_data

,user_id,dt,install_dt,country_code,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,platform,display_campaign_network,acquisition_type,install_build_version,CPE_flag,FTUE_flag,dummy
61871,100186873D03CE09,2026-06-28,2026-06-28,US,2026-06-28,2026-06-01,0,5,3,IOS,Non-Attributed,Non-Attributed,0.78.0,N,A.old,dummy
30611,1008C486C3810433,2026-06-18,2026-06-18,PH,2026-06-14,2026-06-01,0,6,4,IOS,Non-Attributed,Non-Attributed,0.77.0,N,A.old,dummy
30432,1008C486C3810433,2026-06-19,2026-06-18,PH,2026-06-14,2026-06-01,1,8,5,IOS,Non-Attributed,Non-Attributed,0.77.0,N,A.old,dummy
28163,1008C486C3810433,2026-06-20,2026-06-18,PH,2026-06-14,2026-06-01,2,10,7,IOS,Non-Attributed,Non-Attributed,0.77.0,N,A.old,dummy
28898,1008C486C3810433,2026-06-21,2026-06-18,PH,2026-06-14,2026-06-01,3,11,8,IOS,Non-Attributed,Non-Attributed,0.77.0,N,A.old,dummy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44306,FFF7CA7736D2BF38,2026-06-22,2026-06-22,US,2026-06-21,2026-06-01,0,2,1,IOS,FACEBOOK,UA,0.78.0,N,A.old,dummy
43533,FFF7CA7736D2BF38,2026-06-23,2026-06-22,US,2026-06-21,2026-06-01,1,6,4,IOS,FACEBOOK,UA,0.78.0,N,A.old,dummy
97756,FFFC66E056EBEBDF,2026-07-13,2026-07-13,DE,2026-07-12,2026-07-01,0,1,1,AND,Non-Attributed,Non-Attributed,0.79.0,N,B.new,dummy
97197,FFFC66E056EBEBDF,2026-07-14,2026-07-13,DE,2026-07-12,2026-07-01,1,1,1,AND,Non-Attributed,Non-Attributed,0.79.0,N,B.new,dummy


In [75]:
days_since_install_limit = 21

engagement_data = abmetrics[['user_id','dt','days_since_install','n_sessions','n_mins_in_game','n_merges','n_tasks_completed']]
engagement_data = processed_data[['user_id','dt','FTUE_flag','platform']].merge(engagement_data, on=['user_id', 'dt'], how='left')
engagement_data = engagement_data[engagement_data['days_since_install'] <= days_since_install_limit]
engagement_data

,user_id,dt,FTUE_flag,platform,days_since_install,n_sessions,n_mins_in_game,n_merges,n_tasks_completed
0,100186873D03CE09,2026-06-28,A.old,IOS,0,5,39,596.0,15
1,1008C486C3810433,2026-06-18,A.old,IOS,0,13,66,1018.0,23
2,1008C486C3810433,2026-06-19,A.old,IOS,1,21,72,1106.0,14
3,1008C486C3810433,2026-06-20,A.old,IOS,2,19,94,1091.0,13
4,1008C486C3810433,2026-06-21,A.old,IOS,3,13,55,956.0,7
...,...,...,...,...,...,...,...,...,...
94634,FFF7CA7736D2BF38,2026-06-22,A.old,IOS,0,1,6,149.0,2
94635,FFF7CA7736D2BF38,2026-06-23,A.old,IOS,1,8,73,1197.0,22
94636,FFFC66E056EBEBDF,2026-07-13,B.new,AND,0,1,4,16.0,0
94637,FFFC66E056EBEBDF,2026-07-14,B.new,AND,1,1,4,48.0,0


In [76]:
# calculate percentiles P10, P50 and P90 for n_sessions, n_mins_in_game, and n_merges by FTUE_flag, platform, and days_since_install
percentiles = [0.1, 0.5, 0.9]

# Calculate P10, P50, P90 percentiles for n_sessions from engagement_data
percentiles_sessions = engagement_data.groupby(['FTUE_flag', 'platform', 'days_since_install'])['n_sessions'].quantile(percentiles).unstack(fill_value=0).reset_index()
percentiles_sessions.columns = ['FTUE_flag', 'platform', 'days_since_install', 'p10_sessions', 'p50_sessions', 'p90_sessions']
percentiles_sessions


engagement_data_agg = engagement_data.groupby(['FTUE_flag','platform','days_since_install']).agg(
    avg_sessions=('n_sessions', 'mean'),
    avg_n_mins_in_game=('n_mins_in_game', 'mean'),
    avg_n_merges=('n_merges', 'mean'),
    avg_n_tasks_completed=('n_tasks_completed', 'mean'),
    total_users=('user_id', 'nunique')
).reset_index()

engagement_data_agg = engagement_data_agg.merge(percentiles_sessions, on=['FTUE_flag', 'platform', 'days_since_install'], how='left') 

engagement_data_agg.sort_values(['days_since_install','platform','FTUE_flag'], inplace=True)
engagement_data_agg['pctg_diff_avg_sessions'] = engagement_data_agg.groupby(['days_since_install','platform'])['avg_sessions'].pct_change().fillna(0)
engagement_data_agg['pctg_diff_avg_n_mins_in_game'] = engagement_data_agg.groupby(['days_since_install','platform'])['avg_n_mins_in_game'].pct_change().fillna(0)
engagement_data_agg['pctg_diff_avg_n_merges'] = engagement_data_agg.groupby(['days_since_install','platform'])['avg_n_merges'].pct_change().fillna(0)
engagement_data_agg['pctg_diff_avg_n_tasks_completed'] = engagement_data_agg.groupby(['days_since_install','platform'])['avg_n_tasks_completed'].pct_change().fillna(0)

engagement_data_agg

,FTUE_flag,platform,days_since_install,avg_sessions,avg_n_mins_in_game,avg_n_merges,avg_n_tasks_completed,total_users,p10_sessions,p50_sessions,p90_sessions,pctg_diff_avg_sessions,pctg_diff_avg_n_mins_in_game,pctg_diff_avg_n_merges,pctg_diff_avg_n_tasks_completed
0,A.old,AND,0,2.472555,26.941839,338.185387,7.446383,2751,1.0,2.0,5.0,0.0,0.0,0.000000,0.0
42,B.new,AND,0,2.577761,28.609531,377.802364,0.057998,2707,1.0,2.0,5.0,0.042549,0.0619,0.117146,-0.992211
21,A.old,IOS,0,2.47626,25.849572,367.922592,7.976485,8888,1.0,2.0,5.0,0.0,0.0,0.000000,0.0
63,B.new,IOS,0,2.444321,27.038709,403.750776,0.004991,9016,1.0,2.0,5.0,-0.012898,0.046002,0.097380,-0.999374
1,A.old,AND,1,3.930233,31.331843,384.894454,5.991055,1118,1.0,2.0,9.0,0.0,0.0,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,B.new,IOS,19,4.069767,29.104651,410.511628,0.104651,86,1.0,3.0,9.0,-0.079333,-0.182767,0.034182,-0.934686
20,A.old,AND,20,3.88,48.88,440.520000,1.28,25,1.0,3.0,7.0,0.0,0.0,0.000000,0.0
62,B.new,AND,20,4.263158,41.894737,459.210526,0.052632,19,1.0,2.0,8.6,0.098752,-0.142906,0.042428,-0.958882
41,A.old,IOS,20,5.027027,36.567568,400.027027,1.216216,37,1.0,3.0,10.4,0.0,0.0,0.000000,0.0


### Sessions

In [77]:
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='avg_sessions',
              color='FTUE_flag',
              title='Average sessions at day x since install',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()

# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='pctg_diff_avg_sessions',
              color='FTUE_flag',
              title='Average sessions at day x since install (pct change)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()


In [78]:
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='p10_sessions',
              color='FTUE_flag',
              title='10th percentile sessions at day x since install',
              facet_col='platform',
              width=1500,
              height=400,
              hover_data={'p10_sessions': True, 'p50_sessions': True, 'p90_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='p50_sessions',
              color='FTUE_flag',
              title='Median sessions at day x since install',
              facet_col='platform',
              width=1500,
              height=400,
              hover_data={'p10_sessions': True, 'p50_sessions': True, 'p90_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='p90_sessions',
              color='FTUE_flag',
              title='90th percentile sessions at day x since install',
              facet_col='platform',
              width=1500,
              height=400,
              hover_data={'p10_sessions': True, 'p50_sessions': True, 'p90_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()


### Minutes

In [79]:
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='avg_n_mins_in_game',
              color='FTUE_flag',
              title='Average minutes in game at day x since install',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()

# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='pctg_diff_avg_n_mins_in_game',
              color='FTUE_flag',
              title='Average minutes in game at day x since install (pct change)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()

### Merges

In [80]:
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='avg_n_merges',
              color='FTUE_flag',
              title='Average merges at day x since install',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()

# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='pctg_diff_avg_n_merges',
              color='FTUE_flag',
              title='Average merges at day x since install (pct change)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()

### Taks

In [81]:
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='avg_n_tasks_completed',
              color='FTUE_flag',
              title='Average tasks completed at day x since install',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True, 'avg_n_tasks_completed': True},)
fig.show()

# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='pctg_diff_avg_n_tasks_completed',
              color='FTUE_flag',
              title='Average tasks completed at day x since install (pct change)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True, 'avg_n_tasks_completed': True},)
fig.show()

<!-- hide -->  
## Game day reached at day x (work in progress)

Mirrors the level analysis but uses **game days** (in-game calendar progression) instead of levels. Comparing both metrics reveals whether level gates or natural engagement drives pacing — if game days outpace levels, players are replaying content; if levels outpace game days, players are advancing quickly through fewer sessions.

In [82]:
# hide-output
# Compute weighted average game day progression by install cohort and days since install

# Step 1: Count unique users per (install cohort, day since install, max_gameday bucket)
game_day_agg = data.groupby(['install_dt', 'days_since_install', 'max_gameday']).agg(
    unique_users = ('user_id', 'nunique')
    ).reset_index()

# Step 2: Total unique users per cohort-day
daily_cohort_total_users = data.groupby(['install_dt','days_since_install']).agg(
    total_unique_users = ('user_id', 'nunique')
    ).reset_index()

# Step 3: Share of each day's users at each game day value
game_day_agg = game_day_agg.merge(daily_cohort_total_users, on=['install_dt', 'days_since_install'])
game_day_agg['percentage_of_daily_users'] = game_day_agg['unique_users'] / game_day_agg['total_unique_users']

# Step 4: Weighted average game day per cohort-day
weighted_avg = game_day_agg.groupby(['install_dt', 'days_since_install'], group_keys=False).apply(
    lambda x: (x['max_gameday'] * x['unique_users']).sum() / x['unique_users'].sum(),
    include_groups=False
).reset_index()

weighted_avg.columns = ['install_dt', 'days_since_install', 'weighted_avg_max_gameday']
game_day_agg = game_day_agg.merge(weighted_avg, on=['install_dt', 'days_since_install'])

# Drop small buckets (< 50 users) to reduce noise
game_day_agg = game_day_agg[game_day_agg['unique_users'] >= 50]

# Collapse to one row per cohort-day
game_day_agg = game_day_agg.groupby(['install_dt', 'days_since_install']).agg(
    cohort_users = ('unique_users', 'sum'),
    weighted_avg_max_gameday = ('weighted_avg_max_gameday', 'first')
).reset_index()

game_day_agg

,install_dt,days_since_install,cohort_users,weighted_avg_max_gameday
0,2026-06-10,0,526,1.988743
1,2026-06-10,1,142,3.270042
2,2026-06-10,2,50,4.493902
3,2026-06-11,0,442,1.763920
4,2026-06-11,1,129,2.795918
...,...,...,...,...
103,2026-07-19,0,416,2.066038
104,2026-07-19,1,124,6.052083
105,2026-07-20,0,471,1.865424
106,2026-07-20,1,154,2.889796


In [83]:
# hide-output
# Line chart: weighted average game day by install cohort
fig = px.line(game_day_agg, 
              x='days_since_install', 
              y='weighted_avg_max_gameday',
              color='install_dt',
              title='Game day daily progression by cohort',
              width=1200,
              height=600,
              hover_data={'weighted_avg_max_gameday': True, 'cohort_users': True},
              )


fig.show()

In [84]:
# hide-output
# Compute and plot P10/P50/P90 game day distribution by days since install
gameday_dist = data.groupby(['days_since_install', 'max_gameday']).agg(
    users=('user_id', 'count')
).reset_index()

gameday_pcts = gameday_dist.groupby('days_since_install').apply(weighted_quantiles, measure_col='max_gameday', include_groups=False).reset_index()

fig = px.line(
    gameday_pcts.melt(id_vars='days_since_install', var_name='percentile', value_name='max_gameday'),
    x='days_since_install',
    y='max_gameday',
    color='percentile',
    title='Game day distribution by days since install (P10 / P50 / P90)',
    width=1200,
    height=600,
)
fig.show()

In [85]:
# hide-output
# Export notebook to HTML for sharing and archival
export_notebook_html(
    notebook_path='./earlyftue.ipynb',
    output_path='./earlyftue.html',
)

FileNotFoundError: [Errno 2] No such file or directory: 'earlyftue.ipynb'